# Automatic segmentation of OU simulated signals.


We generate a signal that is the sum of OU process to simulate some EEG/ LFP signal. Since we generated the signal we know exact parameter. In normal condition the signal come from real physiology and we would like to estimate the OU process which sum leads to the final signal PSD. The goal of doing so is to use the exact formula of the OU process on an unknown signal to automatically segment its bumps and decay area. 

Here we consequently check the last part of segmentation of bump and decay is feasible once a set of OU processes is fitted. Except instead of fitting OU process we directly use the OU set exact parameters.

In [80]:
%load_ext autoreload
%autoreload 2
%matplotlib tk
import numpy as np
import scipy as sc
import matplotlib.pyplot as plt
from Functions.generate_OU import get_mixed_OU_signals_exact, get_analytical_psd
from Functions.time_frequency import spectrogram

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Generate synthetic signal

In [83]:
# --- Parameters
T = 20 # desired signal duration (s)
dt = 0.001
fs = 1 / dt

lbda_list = [1, 2, 1]
omega_list = [2*np.pi*1, 2*np.pi*10, 2*np.pi*30]
sigma_list = [3, 2, 2]
factor_list = [1, 1, 2]

# --- Generate EEG data
t, y = get_mixed_OU_signals_exact(T, dt, lbda_list, omega_list, sigma_list, factor_list)

# --- compute spectrogram
f_spectro, t_spectro, spectro = spectrogram(y, fs, nfft_factor=2)

# --- Display
fig, axes = plt.subplots(3,  constrained_layout = True)
axes[0].plot(t, y)
axes[0].set_title('Simulated EEG signal')
axes[1].pcolormesh(t_spectro, f_spectro, np.log2(spectro + 1e-11), shading = 'nearest', cmap = 'jet')
axes[1].set_title('Spectrogram')
axes[2].plot(f_spectro, np.log2(np.median(spectro, axis = 1)))
axes[2].set_title('PSD')

axes[1].sharex(axes[0])


plt.show()

# 2. Segmentation of signal bump and decay areas using OU set parameters

As explained in introduction we do not try to fit set of OU to the data but directly use the exact parameters to check the feasibiliy of this segmentation.

In [87]:
f, psd_a = get_analytical_psd(200, 40, lbda_list, omega_list, sigma_list, factor_list)

f_welch, psd_welch = sc.signal.welch(y, fs, nperseg = len(y), average='median')
mask = f_welch <= 40


fig, axes = plt.subplots(3, constrained_layout = True)
axes[0].plot(f_spectro, np.mean(spectro, axis = 1))
axes[0].plot(f, psd_a)
axes[1].semilogy(f_spectro, np.mean(spectro, axis = 1))
axes[1].semilogy(f_spectro, np.median(spectro, axis = 1))
axes[1].semilogy(f, psd_a)
axes[2].semilogy(f_welch[mask], psd_welch[mask])
axes[2].semilogy(f, psd_a)

plt.show()

Not adding the division factor of pi and keeping the multiplicative factor of 2 seems to work when psd is computed by welch and high nperseg.